In [3]:
import pandas as pd

train_df = pd.read_parquet("~/Downloads/qasper-rag/data/train_qasper.parquet") 
test_df = pd.read_parquet("~/Downloads/qasper-rag/data/test_qasper.parquet") 
validation_df = pd.read_parquet("~/Downloads/qasper-rag/data/validation_qasper.parquet")

# print(df.shape)
# print(df.columns.tolist())
# print(df.iloc[0]["title"])
# print(df.iloc[0]["full_text"])

In [4]:
def transform_papers(df):
    records = []

    for _, paper in df.iterrows():
        for section, section_paragraphs in zip(
            paper["full_text"]["section_name"],
            paper["full_text"]["paragraphs"]
        ):
            for paragraph_id, text in enumerate(section_paragraphs):
                text = text.strip()
                if text:
                    records.append({
                        "paper_id": paper["id"],
                        "title": paper["title"],
                        "section": section,
                        "paragraph_id": paragraph_id,
                        "text": text
                    })

    return pd.DataFrame(records, columns=[
        "paper_id", "title", "section", "paragraph_id", "text"
    ])


# Replace train_df, test_df, and validation_df with your existing names.
train_transformed = transform_papers(train_df)
test_transformed = transform_papers(test_df)
validation_transformed = transform_papers(validation_df)

In [5]:
from transformers import AutoTokenizer

# Use this same model when we generate embeddings later.
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_paragraphs(df):
    result = df.copy()

    encoded = tokenizer(
        result["text"].tolist(),
        add_special_tokens=False,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        return_token_type_ids=False
    )

    result["token_ids"] = encoded["input_ids"]
    result["token_count"] = result["token_ids"].map(len)
    return result

train_transformed = tokenize_paragraphs(train_transformed)
validation_transformed = tokenize_paragraphs(validation_transformed)
test_transformed = tokenize_paragraphs(test_transformed)

print(train_transformed[["text", "token_count"]].head())
print(train_transformed["token_count"].describe())

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (888 > 512). Running this sequence through the model will result in indexing errors


                                                text  token_count
0  Affective events BIBREF0 are events that typic...          125
1  Learning affective events is challenging becau...           75
2  In this paper, we propose a simple and effecti...          342
3  We trained the models using a Japanese web cor...           50
4  Learning affective events is closely related t...          144
count    46882.000000
mean       103.818182
std         84.602015
min          1.000000
25%         43.000000
50%         87.000000
75%        143.000000
max       3636.000000
Name: token_count, dtype: float64


In [24]:
def chunk_paragraphs(df, max_tokens=250, overlap=40, max_seq_length=None):
    if not 0 <= overlap < max_tokens:
        raise ValueError("Require 0 <= overlap < max_tokens")
    if max_seq_length is not None and max_tokens > max_seq_length:
        raise ValueError(f"Chunk size {max_tokens} exceeds model limit {max_seq_length}")

    records = []

    for _, row in df.iterrows():
        token_ids = row["token_ids"]

        for chunk_id, start in enumerate(
            range(0, len(token_ids), max_tokens - overlap)
        ):
            end = min(start + max_tokens, len(token_ids))
            chunk_tokens = token_ids[start:end]

            record = row.to_dict()
            record.update({
                "text": tokenizer.decode(
                    chunk_tokens,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=False
                ),
                "token_ids": chunk_tokens,
                "token_count": len(chunk_tokens),
                "chunk_id": chunk_id,
                "token_start": start,
                "token_end": end
            })
            records.append(record)

            if end == len(token_ids):
                break

    return pd.DataFrame(records)

train_chunks = chunk_paragraphs(train_transformed, max_seq_length=model.max_seq_length)
validation_chunks = chunk_paragraphs(validation_transformed, max_seq_length=model.max_seq_length)
test_chunks = chunk_paragraphs(test_transformed, max_seq_length=model.max_seq_length)

for name, chunks in [
    ("train", train_chunks),
    ("validation", validation_chunks),
    ("test", test_chunks)
]:
    print(
        f"{name}: {len(chunks):,} chunks; "
        f"largest = {chunks['token_count'].max()} tokens"
    )

train: 49,569 chunks; largest = 250 tokens
validation: 13,946 chunks; largest = 250 tokens
test: 20,799 chunks; largest = 250 tokens


In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(model_name)

print("Model token limit:", model.max_seq_length)

def embed_chunks(chunks):
    return model.encode(
        chunks["text"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

train_embeddings = embed_chunks(train_chunks)
validation_embeddings = embed_chunks(validation_chunks)
test_embeddings = embed_chunks(test_chunks)

print("Train:", train_embeddings.shape)
print("Validation:", validation_embeddings.shape)
print("Test:", test_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model token limit: 256


Batches:   0%|          | 0/1550 [00:00<?, ?it/s]

Batches:   0%|          | 0/436 [00:00<?, ?it/s]

Batches:   0%|          | 0/650 [00:00<?, ?it/s]

Train: (49569, 384)
Validation: (13946, 384)
Test: (20799, 384)


In [9]:
from pathlib import Path
import numpy as np

Path("artifacts").mkdir(exist_ok=True)

for split, embeddings, chunks in [
    ("train", train_embeddings, train_chunks),
    ("validation", validation_embeddings, validation_chunks),
    ("test", test_embeddings, test_chunks),
]:
    assert len(embeddings) == len(chunks)

    np.save(f"artifacts/{split}_embeddings.npy", embeddings)
    chunks.to_parquet(f"artifacts/{split}_chunks.parquet", index=False)

print("Saved embeddings and matching chunks.")

Saved embeddings and matching chunks.


In [15]:
import faiss
import numpy as np

# FAISS row positions match train_chunks row positions.
vectors = np.ascontiguousarray(train_embeddings, dtype=np.float32)

train_index = faiss.IndexFlatIP(vectors.shape[1])
train_index.add(vectors)

faiss.write_index(train_index, "artifacts/train_index.faiss")

print("Indexed chunks:", train_index.ntotal)

Indexed chunks: 49569


In [16]:
def retrieve(question, top_k=5):
    query_vector = model.encode(
        [question],
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)

    scores, positions = train_index.search(
        query_vector, min(top_k, train_index.ntotal)
    )

    results = train_chunks.iloc[positions[0]].copy()
    results["similarity"] = scores[0]
    return results

results = retrieve("How can discourse relations help identify affective events?")

for _, row in results.iterrows():
    print(f"\nScore: {row['similarity']:.3f}")
    print(f"Paper: {row['title']}")
    print(f"Section: {row['section']}")
    print(row["text"])


Score: 0.779
Paper: Minimally Supervised Learning of Affective Events Using Discourse Relations
Section: Conclusion
in this paper, we proposed to use discourse relations to effectively propagate polarities of affective events from seeds. experiments show that, even with a minimal amount of supervision, the proposed method performed well.

Score: 0.662
Paper: Minimally Supervised Learning of Affective Events Using Discourse Relations
Section: Proposed Method ::: Discourse Relation-Based Event Pairs
our method requires a very small seed lexicon and a large raw corpus. we assume that we can automatically extract discourse - tagged event pairs, $ ( x _ { i1 }, x _ { i2 } ) $ ( $ i = 1, \ cdots $ ) from the raw corpus. we refer to $ x _ { i1 } $ and $ x _ { i2 } $ as former and latter events, respectively. as shown in figure figref1, we limit our scope to two discourse relations : cause and concession.

Score: 0.653
Paper: Minimally Supervised Learning of Affective Events Using Discourse R

In [17]:
validation_vectors = np.ascontiguousarray(
    validation_embeddings, dtype=np.float32
)

validation_index = faiss.IndexFlatIP(validation_vectors.shape[1])
validation_index.add(validation_vectors)

faiss.write_index(
    validation_index, "artifacts/validation_index.faiss"
)

# Inspect the original paper's evaluation annotations.
from pprint import pprint

paper = validation_df.iloc[0]

print("Validation chunks indexed:", validation_index.ntotal)
print("\nPaper:", paper["title"])
print("\nQuestions and answer annotations:")
pprint(paper["qas"])

Validation chunks indexed: 13946

Paper: Cross-lingual Pre-training Based Transfer for Zero-shot Neural Machine Translation

Questions and answer annotations:
{'answers': array([{'answer': array([{'unanswerable': False, 'extractive_spans': array(['BIBREF19', 'BIBREF20'], dtype=object), 'yes_no': None, 'free_form_answer': '', 'evidence': array(['Table TABREF19 and TABREF26 report zero-shot results on Europarl and Multi-UN evaluation sets, respectively. We compare our approaches with related approaches of pivoting, multilingual NMT (MNMT) BIBREF19, and cross-lingual transfer without pretraining BIBREF16. The results show that our approaches consistently outperform other approaches across languages and datasets, especially surpass pivoting, which is a strong baseline in the zero-shot scenario that multilingual NMT systems often fail to beat BIBREF19, BIBREF20, BIBREF23. Pivoting translates source to pivot then to target in two steps, causing inefficient translation process. Our approaches

In [18]:
# Select the first validation paper and its first question.
paper = validation_df.iloc[0]
question = paper["qas"]["question"][0]

# Search only this paper's chunks, using existing embeddings.
positions = np.flatnonzero(
    validation_chunks["paper_id"].to_numpy() == paper["id"]
)

paper_index = faiss.IndexFlatIP(validation_embeddings.shape[1])
paper_index.add(np.ascontiguousarray(
    validation_embeddings[positions], dtype=np.float32
))

query_vector = model.encode(
    [question],
    normalize_embeddings=True,
    convert_to_numpy=True
)

scores, local_positions = paper_index.search(
    np.ascontiguousarray(query_vector, dtype=np.float32),
    min(5, len(positions))
)

results = validation_chunks.iloc[
    positions[local_positions[0]]
].copy()
results["similarity"] = scores[0]

print("QUESTION:", question)

for rank, (_, row) in enumerate(results.iterrows(), start=1):
    print(f"\n[{rank}] Score: {row['similarity']:.3f}")
    print("Section:", row["section"])
    print(row["text"])

# Show gold evidence separately; it never enters the search index.
gold_evidence = set()

for answer in paper["qas"]["answers"][0]["answer"]:
    if not answer["unanswerable"]:
        gold_evidence.update(answer["evidence"])

print("\nGOLD EVIDENCE:")
for evidence in sorted(gold_evidence):
    print("\n", evidence)

QUESTION: which multilingual approaches do they compare with?

[1] Score: 0.540
Section: Experiments ::: Setup ::: Datasets.
for multiun corpus, we use four languages : english ( en ) is set as the pivot language, which has parallel data with other three languages which do not have parallel data between each other. the three languages are arabic ( ar ), spanish ( es ), and russian ( ru ), and mutual translation between themselves constitutes six zero - shot translation direction for evaluation. we use 80k bpe splits as the vocabulary. note that all sentences are tokenized by the tokenize. perl script, and we lowercase all data to avoid a large vocabulary for the multiun corpus.

[2] Score: 0.440
Section: Experiments ::: Setup
we evaluate our cross - lingual pre - training based transfer approach against several strong baselines on two public datatsets, europarl bibref31 and multiun bibref32, which contain multi - parallel evaluation data to assess the zero - shot performance. in all ex

In [19]:
scores, local_positions = paper_index.search(
    np.ascontiguousarray(query_vector, dtype=np.float32),
    min(10, len(positions))
)

results = validation_chunks.iloc[
    positions[local_positions[0]]
].copy()
results["similarity"] = scores[0]

for rank, (_, row) in enumerate(results.iterrows(), start=1):
    print(f"\n[{rank}] Score: {row['similarity']:.3f}")
    print("Section:", row["section"])
    print(row["text"])


[1] Score: 0.540
Section: Experiments ::: Setup ::: Datasets.
for multiun corpus, we use four languages : english ( en ) is set as the pivot language, which has parallel data with other three languages which do not have parallel data between each other. the three languages are arabic ( ar ), spanish ( es ), and russian ( ru ), and mutual translation between themselves constitutes six zero - shot translation direction for evaluation. we use 80k bpe splits as the vocabulary. note that all sentences are tokenized by the tokenize. perl script, and we lowercase all data to avoid a large vocabulary for the multiun corpus.

[2] Score: 0.440
Section: Experiments ::: Setup
we evaluate our cross - lingual pre - training based transfer approach against several strong baselines on two public datatsets, europarl bibref31 and multiun bibref32, which contain multi - parallel evaluation data to assess the zero - shot performance. in all experiments, we use bleu as the automatic metric for translation

In [20]:
paper_chunks = validation_chunks.iloc[positions]

matches = paper_chunks[
    paper_chunks["text"].str.contains(
        "we compare our approaches", case=False, regex=False
    )
]

print(matches[["section", "text"]].to_string(index=False))

                     section                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              text
Experimen

In [23]:
import re
from rank_bm25 import BM25Okapi

def bm25_tokens(text):
    return re.findall(r"\b\w+\b", text.lower())

paper_chunks = validation_chunks.iloc[positions].copy()

bm25 = BM25Okapi([
    bm25_tokens(text)
    for text in paper_chunks["text"]
])

bm25_scores = bm25.get_scores(bm25_tokens(question))
top_positions = np.argsort(-bm25_scores)[:5]

bm25_results = paper_chunks.iloc[top_positions].copy()
bm25_results["bm25_score"] = bm25_scores[top_positions]

print("QUESTION:", question)

for rank, (_, row) in enumerate(bm25_results.iterrows(), start=1):
    print(f"\n[{rank}] BM25: {row['bm25_score']:.3f}")
    print("Section:", row["section"])
    print(row["text"])

QUESTION: which multilingual approaches do they compare with?

[1] BM25: 9.669
Section: Experiments ::: Main Results
table tabref19 and tabref26 report zero - shot results on europarl and multi - un evaluation sets, respectively. we compare our approaches with related approaches of pivoting, multilingual nmt ( mnmt ) bibref19, and cross - lingual transfer without pretraining bibref16. the results show that our approaches consistently outperform other approaches across languages and datasets, especially surpass pivoting, which is a strong baseline in the zero - shot scenario that multilingual nmt systems often fail to beat bibref19, bibref20, bibref23. pivoting translates source to pivot then to target in two steps, causing inefficient translation process. our approaches use one encoder - decoder model to translate between any zero - shot directions, which is more efficient than pivoting. regarding the comparison between transfer approaches, our cross - lingual pretraining based transfe

In [33]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

def normalize_text(text):
    return " ".join(text.split())

evaluation_rows = []
skipped = 0
unmapped_evidence = 0

for _, paper in tqdm(
    validation_df.iterrows(), total=len(validation_df)
):
    positions = np.flatnonzero(
        validation_chunks["paper_id"].to_numpy() == paper["id"]
    )
    if len(positions) == 0:
        continue

    chunks = validation_chunks.iloc[positions]
    chunk_keys = list(zip(chunks["section"], chunks["paragraph_id"]))

    # Match gold evidence to ORIGINAL paragraphs, before token decoding.
    paragraph_lookup = {}
    for section, paragraphs in zip(
        paper["full_text"]["section_name"],
        paper["full_text"]["paragraphs"]
    ):
        for paragraph_id, text in enumerate(paragraphs):
            paragraph_lookup.setdefault(
                normalize_text(text), set()
            ).add((section, paragraph_id))

    bm25 = BM25Okapi([
        bm25_tokens(text) for text in chunks["text"]
    ])
    paper_vectors = np.ascontiguousarray(
        validation_embeddings[positions], dtype=np.float32
    )

    questions = list(paper["qas"]["question"])
    query_vectors = model.encode(
        questions,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    for i, question in enumerate(questions):
        gold_keys = set()

        for answer in paper["qas"]["answers"][i]["answer"]:
            if answer["unanswerable"]:
                continue

            for evidence in answer["evidence"]:
                matches = paragraph_lookup.get(
                    normalize_text(evidence), set()
                )
                if matches:
                    gold_keys.update(matches)
                else:
                    unmapped_evidence += 1

        # Excludes unanswerable questions and evidence we cannot map.
        if not gold_keys:
            skipped += 1
            continue

        dense_scores = paper_vectors @ query_vectors[i]
        keyword_scores = bm25.get_scores(bm25_tokens(question))

        for method, scores in [
            ("embedding", dense_scores),
            ("BM25", keyword_scores)
        ]:
            ranking = np.argsort(-scores, kind="stable")

            record = {
                "paper_id": paper["id"],
                "question": question,
                "method": method
            }

            for k in [1, 5, 10]:
                retrieved_keys = {
                    chunk_keys[j] for j in ranking[:k]
                }
                record[f"hit@{k}"] = int(
                    bool(retrieved_keys & gold_keys)
                )

            evaluation_rows.append(record)

evaluation = pd.DataFrame(evaluation_rows)

print(evaluation.groupby("method")[
    ["hit@1", "hit@5", "hit@10"]
].mean().round(3))

print("\nQuestions evaluated:", len(evaluation) // 2)
print("Questions skipped:", skipped)
print("Unmapped evidence entries:", unmapped_evidence)

evaluation.to_csv(
    "artifacts/validation_retrieval_evaluation.csv", index=False
)

  0%|          | 0/281 [00:00<?, ?it/s]

           hit@1  hit@5  hit@10
method                         
BM25       0.213  0.581   0.753
embedding  0.291  0.654   0.798

Questions evaluated: 888
Questions skipped: 117
Unmapped evidence entries: 382


In [34]:
from rank_bm25 import BM25Okapi
import faiss
import numpy as np

# --- Index building ---

def build_bm25(chunks):
    tokenized = [text.lower().split() for text in chunks["text"]]
    return BM25Okapi(tokenized)

def build_faiss(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)  # Inner product; works because embeddings are normalized
    index.add(embeddings)
    return index

train_bm25 = build_bm25(train_chunks)
train_faiss = build_faiss(train_embeddings)

# --- Reciprocal Rank Fusion ---

def rrf_score(rank, k=60):
    return 1 / (k + rank)

def hybrid_search(query, chunks, bm25_index, faiss_index, query_embedding, top_k=10):
    n = len(chunks)

    # BM25 ranking
    bm25_scores = bm25_index.get_scores(query.lower().split())
    bm25_ranks = np.argsort(bm25_scores)[::-1]  # highest score = rank 0

    # Dense ranking
    D, I = faiss_index.search(query_embedding.reshape(1, -1), n)
    dense_ranks = I[0]  # indices in order of similarity

    # RRF fusion
    rrf = np.zeros(n)
    for rank, idx in enumerate(bm25_ranks):
        rrf[idx] += rrf_score(rank)
    for rank, idx in enumerate(dense_ranks):
        rrf[idx] += rrf_score(rank)

    top_indices = np.argsort(rrf)[::-1][:top_k]
    results = chunks.iloc[top_indices].copy()
    results["rrf_score"] = rrf[top_indices]
    return results.reset_index(drop=True)

# --- Example usage ---

query = "What optimizer was used for training?"
query_embedding = model.encode(query, normalize_embeddings=True, convert_to_numpy=True)

results = hybrid_search(
    query=query,
    chunks=train_chunks,
    bm25_index=train_bm25,
    faiss_index=train_faiss,
    query_embedding=query_embedding,
    top_k=5
)

print(results[["title", "section", "text", "rrf_score"]])

                                               title  \
0  HuggingFace's Transformers: State-of-the-art N...   
1  A Robust Hybrid Approach for Textual Document ...   
2  Exploring Multilingual Syntactic Sentence Repr...   
3  Hybrid Code Networks: practical and efficient ...   
4     A Multi-Turn Emotionally Engaging Dialog Model   

                                       section  \
0                  Library design ::: Training   
1                           Experimental Setup   
2                          Method ::: Training   
3                 Model implementation details   
4  Evaluation ::: Baselines and Implementation   

                                                text  rrf_score  
0  optimizer - the library provides a few optimiz...   0.032796  
1  rmsprop is used as an optimizer with learning ...   0.030835  
2  the upos targets were obtained using standford...   0.028161  
3            optimizer = adadelta ( clipnorm = 1. ),   0.026511  
4  we optimized the objective fu

In [35]:
from tqdm import tqdm

# --- Evaluation ---
# Assumes you have a list of (query, gold_chunk_ids) pairs.
# gold_chunk_ids should match the (paper_id, paragraph_id, chunk_id) 
# tuples in your chunks dataframe.

def recall_at_k(results, gold_ids, k):
    retrieved_ids = set(
        zip(
            results.iloc[:k]["paper_id"],
            results.iloc[:k]["paragraph_id"],
            results.iloc[:k]["chunk_id"]
        )
    )
    return int(bool(retrieved_ids & gold_ids))

def evaluate_retrieval(queries, chunks, bm25_index, faiss_index, ks=(1, 3, 5, 10)):
    recalls = {k: [] for k in ks}

    for query_text, gold_ids in tqdm(queries, desc="Evaluating"):
        query_embedding = model.encode(
            query_text,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        results = hybrid_search(
            query=query_text,
            chunks=chunks,
            bm25_index=bm25_index,
            faiss_index=faiss_index,
            query_embedding=query_embedding,
            top_k=max(ks)
        )

        for k in ks:
            recalls[k].append(recall_at_k(results, gold_ids, k))

    return {f"Recall@{k}": round(sum(v) / len(v), 4) for k, v in recalls.items()}


# --- Build query/gold pairs from QASPER ---
# QASPER answers reference paper_id + evidence spans; map those to chunk ids.

def build_query_gold_pairs(qa_df, chunks):
    pairs = []

    for _, row in qa_df.iterrows():
        paper_id = row["id"]
        paper_chunks = chunks[chunks["paper_id"] == paper_id]

        for question, answers in zip(row["qas"]["question"], row["qas"]["answers"]):
            # Collect all evidence paragraph indices across annotators
            gold_paragraph_ids = set()
            for answer_list in answers["answer"]:
                for evidence in answer_list.get("evidence", []):
                    # Match evidence text to a paragraph_id in our chunks
                    matched = paper_chunks[
                        paper_chunks["text"].str.contains(
                            evidence[:80],  # match on first 80 chars to avoid regex issues
                            regex=False,
                            case=False
                        )
                    ]
                    for _, chunk_row in matched.iterrows():
                        gold_paragraph_ids.add((
                            chunk_row["paper_id"],
                            chunk_row["paragraph_id"],
                            chunk_row["chunk_id"]
                        ))

            if gold_paragraph_ids:
                pairs.append((question, gold_paragraph_ids))

    return pairs


# Build pairs and run eval on validation set
val_pairs = build_query_gold_pairs(validation_df, validation_chunks)

print(f"Evaluating on {len(val_pairs)} questions...")
metrics = evaluate_retrieval(
    queries=val_pairs,
    chunks=validation_chunks,
    bm25_index=build_bm25(validation_chunks),
    faiss_index=build_faiss(validation_embeddings),
)

for metric, score in metrics.items():
    print(f"  {metric}: {score}")

Evaluating on 601 questions...


Evaluating: 100%|██████████| 601/601 [01:22<00:00,  7.29it/s]

  Recall@1: 0.1148
  Recall@3: 0.2246
  Recall@5: 0.2862
  Recall@10: 0.3677


In [31]:
#Claude Code LLM Assist 
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer(model_name, device=device)

Using device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [36]:
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "BAAI/bge-base-en-v1.5"
model = SentenceTransformer(model_name, device=device)

print("New model token limit:", model.max_seq_length)

train_embeddings = embed_chunks(train_chunks)
validation_embeddings = embed_chunks(validation_chunks)
test_embeddings = embed_chunks(test_chunks)

print("Train:", train_embeddings.shape)
print("Validation:", validation_embeddings.shape)
print("Test:", test_embeddings.shape)

# Save embeddings and chunks to disk — prevents re-running on kernel restart
np.save("train_embeddings_bge.npy", train_embeddings)
np.save("validation_embeddings_bge.npy", validation_embeddings)
np.save("test_embeddings_bge.npy", test_embeddings)

train_chunks.to_parquet("train_chunks.parquet")
validation_chunks.to_parquet("validation_chunks.parquet")
test_chunks.to_parquet("test_chunks.parquet")

print("Embeddings and chunks saved.")

Using device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

New model token limit: 512


Batches:   0%|          | 0/1550 [00:00<?, ?it/s]

KeyboardInterrupt: 